# Lasso cells on a UMAP, pseudobulk them onto the genome

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/14_scatac_umap.ipynb)

Single-cell ATAC gives every cell a few thousand fragments, which is nearly all zero on its own. Pooling a group of cells into one profile is what makes a track, and the group is usually a cluster someone drew on a UMAP. Here the UMAP stays live: lasso cells and the genome track repaints with the pseudobulk of exactly those cells.

The data is the 10x **5k PBMC** scATAC experiment, annotated by [SnapATAC2](https://scverse.org/SnapATAC2/). Neither file is downloaded — the UMAP and the labels are read out of an 837 MB `.h5ad` by HTTP range, and the fragments are range-queried out of a 1 GB tabix-indexed file.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy h5py plotly

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## The cells, without downloading the h5ad

An `.h5ad` is HDF5, and HDF5 seeks. Give `h5py` a file object that turns seeks into `Range` requests and reading `obsm/X_umap`, `obs/cell_type` and `obs/index` costs a handful of requests against a file you never fetch. `snapatac2` is not needed to read it, and is not installed here — this is a spec-compliant h5ad.

In [ ]:
import io
import urllib.request

import h5py
import numpy as np

H5 = (
    "https://exampledata.scverse.org/snapatac2/"
    "atac_pbmc_5k_annotated.h5ad"
)
# The CDN 403s urllib's default agent, and answers HEAD with the same.
UA = {"User-Agent": "Mozilla/5.0"}


class RangeReader(io.RawIOBase):
    """A seekable file over HTTP, so h5py reads only what it needs."""

    def __init__(self, url):
        self.url = url
        self.pos = 0
        self.requests = 0
        head = self._get(0, 0)
        self.size = int(head.headers["Content-Range"].split("/")[1])

    def _get(self, start, end):
        req = urllib.request.Request(
            self.url, headers={**UA, "Range": f"bytes={start}-{end}"}
        )
        self.requests += 1
        return urllib.request.urlopen(req)

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        base = (0, self.pos, self.size)[whence]
        self.pos = base + offset
        return self.pos

    def readinto(self, buf):
        if self.pos >= self.size:
            return 0
        end = min(self.pos + len(buf), self.size) - 1
        data = self._get(self.pos, end).read()
        buf[: len(data)] = data
        self.pos += len(data)
        return len(data)


raw = RangeReader(H5)
h5 = h5py.File(io.BufferedReader(raw, 1 << 20), "r")

umap = h5["obsm/X_umap"][:]
cell_type_group = h5["obs/cell_type"]
categories = np.array(
    [c.decode() for c in cell_type_group["categories"][:]]
)
labels = categories[cell_type_group["codes"][:]]
barcodes = np.array([b.decode() for b in h5["obs/index"][:]])

print(f"{len(barcodes):,} cells, {len(categories)} labels, "
      f"{raw.requests} range requests against {raw.size / 1e6:.0f} MB")

## The fragments, also without downloading them

10x publishes the fragments as a tabix-indexed BED, so `pysam` fetches one window out of a gigabyte. The window here is **MS4A1**, a B-cell surface marker, which is the point: whether its promoter is open should depend on which cells you pooled.

htslib needs `CURL_CA_BUNDLE` pointed at a CA bundle before it will read an https URL.

In [ ]:
import os

import certifi

os.environ["CURL_CA_BUNDLE"] = certifi.where()

import pysam  # noqa: E402 — must follow the CA bundle

FRAGMENTS = (
    "https://cf.10xgenomics.com/samples/cell-atac/2.0.0/"
    "atac_pbmc_5k_nextgem/atac_pbmc_5k_nextgem_fragments.tsv.gz"
)
CHROM, START, END = "chr11", 60_450_000, 60_480_000  # MS4A1, hg38

tabix = pysam.TabixFile(FRAGMENTS)
rows = [r.split("\t") for r in tabix.fetch(CHROM, START, END)]
frag_start = np.array([int(r[1]) for r in rows])
frag_end = np.array([int(r[2]) for r in rows])
frag_barcode = np.array([r[3] for r in rows])

print(f"{len(rows):,} fragments in the window")

## Pseudobulk of a selection

Both ends of a fragment are a cut site, so binning starts and ends together is the profile. Dividing by the number of cells selected is what makes two selections comparable — without it a lasso around a big cluster always looks like more signal.

The column is named `score`, which is what turns the result into a real wiggle with a value axis rather than boxes to color by hand.

In [ ]:
BIN = 200
edges = np.arange(START, END + BIN, BIN)


def pseudobulk(selected):
    """Cut sites per 1000 cells, in BIN bp bins, for a barcode set."""
    keep = np.isin(frag_barcode, selected)
    cuts = np.histogram(frag_start[keep], bins=edges)[0]
    cuts = cuts + np.histogram(frag_end[keep], bins=edges)[0]
    per_1k = cuts / max(len(selected), 1) * 1000
    return pd.DataFrame(
        {
            "chrom": CHROM,
            "start": edges[:-1],
            "end": edges[:-1] + BIN,
            "score": per_1k.round(1),
        }
    )


import pandas as pd  # noqa: E402

for label in ["Naive B", "Memory B", "CD14 Mono", "NK"]:
    sel = barcodes[labels == label]
    print(f"{label:<10} {len(sel):>4} cells  "
          f"peak {pseudobulk(sel).score.max():>6.1f} cuts/1k cells")

## The UMAP, the view, and the wire between them

A plotly `FigureWidget` has a lasso and an `on_selection` callback. The callback hands back point indices, which index straight into `barcodes` — so the whole wire is one function that replaces the track.

The opening selection is set in code rather than by gesture, so the notebook shows something the moment it runs (and so it renders headless).

In [ ]:
import plotly.graph_objects as go
from ipywidgets import VBox

from jbrowse_anywidget import LinearGenomeView, features_track, fetch_hub

hg38 = fetch_hub("hg38")
view = LinearGenomeView(
    assembly=hg38["assemblies"][0],
    aggregateTextSearchAdapters=hg38["aggregateTextSearchAdapters"],
    location=f"{CHROM}:{START:,}..{END:,}",
)

scatter = go.FigureWidget(
    [
        go.Scattergl(
            x=umap[labels == c, 0],
            y=umap[labels == c, 1],
            mode="markers",
            name=c,
            customdata=np.where(labels == c)[0],
            marker={"size": 4},
        )
        for c in categories
    ]
)
scatter.update_layout(
    dragmode="lasso", height=420, margin={"l": 0, "r": 0, "t": 0, "b": 0}
)


def show(indices, name):
    view.update(
        tracks=[
            features_track(
                pseudobulk(barcodes[indices]),
                name=f"{name} ({len(indices)} cells)",
                track_id="pseudobulk",
                color="#4682b4",
            )
        ]
    )


def on_lasso(trace, points, state):
    picked = np.concatenate(
        [t.customdata[t.selectedpoints or []] for t in scatter.data]
    ).astype(int)
    if len(picked):
        show(picked, "lassoed cells")


for trace in scatter.data:
    trace.on_selection(on_lasso)

b_cells = np.where(np.isin(labels, ["Naive B", "Memory B"]))[0]
show(b_cells, "B cells")

VBox([scatter, view])

## What to try

Lasso the two B clusters and the MS4A1 promoter carries a peak; lasso the monocytes — a *larger* group — and it flattens. That is the check that the track is following the selection rather than the cell count.

The window is fetched once and every reselection is arithmetic over what is already in memory, which is what keeps the lasso live. Panning to another gene is one more `tabix.fetch`; [notebook 10](10_region_reactive.ipynb) wires that to the view's own location, and [notebook 13](13_large_wiggle.ipynb) is where to go when the answer stops fitting in `features_track`.